# 🐳 Docker 简介与基本操作

## 一、什么是 Docker？

**Docker** 是一个开源的容器化平台，用于**构建、打包、部署**应用程序。它利用操作系统级别的虚拟化技术，将应用及其依赖打包到一个轻量级、可移植的**容器**中。

### 核心概念

| 概念 | 说明 |
|------|------|
| **镜像（Image）** | 一个只读的模板，包含运行应用所需的代码、运行时、库、环境变量和配置文件。 |
| **容器（Container）** | 镜像的运行实例，是一个独立、隔离的轻量级进程。 |
| **Dockerfile** | 一个文本文件，包含构建镜像的指令。 |
| **仓库（Repository）** | 存放镜像的地方，如 Docker Hub。 |
| **卷（Volume）** | 持久化容器数据的机制。 |

### Docker vs 传统虚拟机

| 特性 | Docker 容器 | 传统虚拟机 |
|------|------------|-----------|
| 启动速度 | 秒级 | 分钟级 |
| 系统资源 | 共享宿主机内核，占用小 | 每个 VM 包含完整 OS，占用大 |
| 隔离性 | 进程级隔离 | 硬件级虚拟化，隔离更强 |

---

## 二、Docker 基本操作

### 1️⃣ 镜像操作

```bash
# 拉取镜像
docker pull nginx:latest

# 查看本地镜像
docker images

# 删除镜像
docker rmi nginx:latest

# 搜索镜像
docker search nginx
```

### 2️⃣ 容器操作

```bash
# 创建并启动容器
docker run -d --name my-nginx -p 8080:80 nginx:latest

# 查看运行中的容器
docker ps

# 查看所有容器（含已停止）
docker ps -a

# 停止容器
docker stop my-nginx

# 启动已停止的容器
docker start my-nginx

# 进入容器内部
docker exec -it my-nginx /bin/bash

# 查看容器日志
docker logs my-nginx

# 删除容器
docker rm my-nginx
```

### 3️⃣ Dockerfile 示例

```dockerfile
# 使用官方 Python 镜像作为基础
FROM python:3.9-slim

# 设置工作目录
WORKDIR /app

# 复制依赖文件
COPY requirements.txt .

# 安装依赖
RUN pip install --no-cache-dir -r requirements.txt

# 复制源代码
COPY . .

# 暴露端口
EXPOSE 5000

# 启动命令
CMD ["python", "app.py"]
```

构建并运行：

```bash
docker build -t my-app .
docker run -d -p 5000:5000 my-app
```

### 4️⃣ 数据持久化（Volume）

```bash
# 创建卷
docker volume create my-volume

# 挂载卷到容器
docker run -d -v my-volume:/data --name my-container nginx:latest

# 挂载宿主机目录（绑定挂载）
docker run -d -v /host/path:/container/path nginx:latest
```

### 5️⃣ 网络操作

```bash
# 列出网络
docker network ls

# 创建自定义网络
docker network create my-network

# 将容器连接到网络
docker network connect my-network my-container
```

### 6️⃣ Docker Compose（多容器管理）

`docker-compose.yml` 示例：

```yaml
version: '3'
services:
  web:
    build: .
    ports:
      - "5000:5000"
  redis:
    image: redis:alpine
```

```bash
# 启动所有服务
docker-compose up -d

# 停止并删除
docker-compose down
```

---

## 三、常用命令速查表

| 命令 | 用途 |
|------|------|
| `docker pull <image>` | 拉取镜像 |
| `docker build -t <tag> .` | 构建镜像 |
| `docker run <image>` | 运行容器 |
| `docker ps` | 查看运行中的容器 |
| `docker stop <container>` | 停止容器 |
| `docker rm <container>` | 删除容器 |
| `docker rmi <image>` | 删除镜像 |
| `docker logs <container>` | 查看容器日志 |
| `docker exec -it <container> bash` | 进入容器 |
| `docker-compose up -d` | 启动所有服务 |

---

> **总结**：Docker 让应用的开发、交付和部署变得更加标准化和高效。掌握以上基本操作，就能轻松上手日常的容器化工作。 ✅

# 🐟 小金鱼识别项目总结

## 一、项目概述

基于 **YOLOv3** 目标检测算法，实现对鱼缸视频中三种金鱼的实时检测与分类：**红金鱼 (red)**、**黑金鱼 (black)**、**白金鱼 (white)**。

---

## 二、项目结构

```
week10/
├── 图片文件/          # 85张训练图片 (416×416)
├── 标注文件/          # 85个 Pascal VOC XML 标注
├── 1.mp4              # 待推理的测试视频 (960×544, 427帧)
├── train_yolo_xml.py  # 训练脚本
├── detect_mp4.py      # 推理脚本
├── checkpoints/       # 保存的模型权重
└── output_detected.mp4# 输出检测结果视频
```

## 三、技术栈

| 组件 | 技术选型 |
|------|---------|
| 目标检测框架 | YOLOv3 (Darknet-53 + FPN) |
| 深度学习框架 | PyTorch 2.5 |
| 编程语言 | Python 3.11 |
| 数据标注格式 | Pascal VOC XML |
| GPU 加速 | CUDA 12.1 |

## 四、关键技术点

### 1. Anchor 设计
- 使用 **K-Means 聚类** 对 447 个标注框计算得到 9 个 Anchor
- 按面积分配到 3 个检测尺度：52×52（小目标）、26×26（中目标）、13×13（大目标）

### 2. 标签分配
- 跨尺度全局选择 **IoU 最高** 的 Anchor 作为正样本
- 引入 **Ignore 区域**：IoU > 0.5 但不够正样本的 Anchor 不参与负样本 Loss

### 3. 损失函数
- `reduction="sum"` 避免负样本被海量稀释
- **BCE for 分类**（与推理时 sigmoid 一致）
- **Sigmoid on xy**：预测坐标偏移先过 sigmoid 再与 target 算 MSE
- 正样本权重 15.0，负样本权重 1.0

### 4. 推理后处理
- 向量化解码（`torch.where`）代替逐 cell 循环
- 分类用 **sigmoid**（YOLOv3 多标签），不用 softmax
- `torchvision.ops.nms` 内置 NMS
- 用 `obj_conf` 直接做置信度过滤（不乘 cls_max）

## 五、训练参数

| 参数 | 值 |
|------|----|
| 训练集 | 73 张 |
| 验证集 | 12 张 |
| Epochs | 300 |
| Batch Size | 8 |
| 学习率 | 0.001 |
| 优化器 | Adam |
| 类别数 | 3 (red/black/white) |
| 输入尺寸 | 416×416 |

## 六、检测效果

```
每帧约 10~15 个框
├── red   ≈ 9 个
├── black ≈ 2 个
└── white ≈ 2 个

阈值: obj_conf > 0.58, NMS > 0.45
```

## 七、改进历程

| 版本 | 主要问题 | 解决方案 |
|------|---------|---------|
| v1 | 损失函数负样本用 `mean` 被稀释 | 改为 `sum` |
| v2 | Anchor 太大，只用到 52 尺度 | K-Means 重新聚类 |
| v3 | 框太多（~100个/帧） | 分类改 sigmoid，obj_conf 直出 |
| v4 | 只检出红鱼 | 返回 cls_ids，阈值调平 |
| v5 | 位置错乱 | 修复 grid 索引计算 |
| v6 | 白框太多 | 调高阈值 + ignore 区域 |

## 八、已知局限

- **训练数据仅 85 张**，模型过拟合严重（Val Loss 约 Epoch 25 触底反弹）
- **13/26 尺度几乎无检测**，Anchor 分配仍需优化
- 模型 `obj_conf` 集中在中段（0.55~0.62），缺乏置信度区分度
- 改进方向：增加数据量、数据增强（Mosaic）、标签平滑、参考代码的 `reduction='sum'` 损失